# 📱 Viral Video Finder — No API Key Mode
Press **Runtime → Run all**. The agent finds public-license Wikimedia Commons videos and downloads a ZIP to your phone. This is **not** a YouTube/Instagram viral-video downloader. It saves a license manifest. If no suitable files are found, ZIP contains the results and explanation.


In [ ]:
# Viral Video Agent (mobile, no API keys): Wikimedia Commons licensed clips
# Output: videos.zip, source/license manifest, and optional YouTube search links.
from pathlib import Path
from urllib.parse import urlparse, quote
from google.colab import files
import requests, subprocess, zipfile, json, re, html

TOPIC = "gaming"   # change to nature, cities, technology, etc.
MAX_VIDEOS = 3
MAX_BYTES = 38 * 1024 * 1024
base = Path("/content/viral_agent_results")
clips = base / "videos"
clips.mkdir(parents=True, exist_ok=True)
headers = {"User-Agent": "EducationalLicensedVideoFinder/1.0 (Google Colab example)"}
candidates = [
    f"filetype:video {TOPIC}",
    "filetype:video video game" if TOPIC == "gaming" else f"filetype:video {TOPIC} footage",
    "filetype:video nature",
    "filetype:video time lapse",
]
def clean(s):
    return html.unescape(re.sub(r"<[^>]*>", " ", str(s or ""))).strip()

results, seen = [], set()
for term in candidates:
    if len(results) >= MAX_VIDEOS:
        break
    try:
        response = requests.get("https://commons.wikimedia.org/w/api.php",
            params={"action":"query","format":"json","origin":"*","generator":"search",
                    "gsrsearch":term,"gsrnamespace":6,"gsrlimit":40,
                    "prop":"imageinfo","iiprop":"url|size|mime|extmetadata"},
            headers=headers,timeout=30)
        response.raise_for_status()
        pages = (response.json().get("query") or {}).get("pages",{})
    except Exception as e:
        print("Search error:", type(e).__name__)
        continue
    for p in pages.values():
        i = (p.get("imageinfo") or [{}])[0]
        url, mime = i.get("url",""), i.get("mime","")
        if not url or url in seen or urlparse(url).hostname != "upload.wikimedia.org":
            continue
        if not mime.startswith("video/") and not url.lower().endswith((".webm",".ogv",".mp4")):
            continue
        if int(i.get("size") or 0) > MAX_BYTES:
            continue
        meta = i.get("extmetadata") or {}
        value = lambda k: clean((meta.get(k) or {}).get("value",""))
        license_name = value("LicenseShortName")
        if not re.search(r"CC0|CC\s*BY|public\s*domain|\bPD\b",license_name,re.I):
            continue
        seen.add(url)
        number = len(results)+1
        suffix = ".mp4" if mime == "video/mp4" else ".webm" if mime == "video/webm" else ".ogv"
        original = clips/f"video_{number}{suffix}"
        try:
            with requests.get(url,headers=headers,stream=True,timeout=(15,120)) as r:
                r.raise_for_status()
                total=0
                with original.open("wb") as out:
                    for chunk in r.iter_content(256*1024):
                        total+=len(chunk)
                        if total>MAX_BYTES: raise ValueError("file too large")
                        out.write(chunk)
            final=original
            if suffix!=".mp4":
                mp4=clips/f"video_{number}.mp4"
                try:
                    proc=subprocess.run(["ffmpeg","-nostdin","-y","-loglevel","error",
                        "-i",str(original),"-c:v","libx264","-crf","27",
                        "-preset","veryfast","-pix_fmt","yuv420p","-c:a","aac",
                        str(mp4)],timeout=180,capture_output=True)
                    if proc.returncode==0 and mp4.exists() and mp4.stat().st_size:
                        original.unlink()
                        final=mp4
                except Exception: pass
            title=p.get("title","")
            results.append({"title":title,"saved_as":final.name,
                "source":"https://commons.wikimedia.org/wiki/"+quote(title.replace(" ","_"),safe=":/_()"),
                "creator":value("Artist"),"license":license_name,
                "license_url":value("LicenseUrl"),"attribution":value("Attribution")})
            print("✅",final.name,"|",license_name,"|",title[:65])
            if len(results)>=MAX_VIDEOS: break
        except Exception as e:
            original.unlink(missing_ok=True)
            print("Skipped:",type(e).__name__)

(base/"SOURCES_AND_LICENSES.json").write_text(json.dumps(results,ensure_ascii=False,indent=2))
(base/"README.txt").write_text(
    "These are Wikimedia Commons openly licensed videos, NOT verified viral YouTube/Instagram clips.\n"
    "Check source and creator-license terms before republishing, and provide required attribution.\n"
    "YouTube trending search: https://www.youtube.com/results?search_query=GTA+6+Shorts\n"
    "Marvel: https://www.youtube.com/results?search_query=Marvel+Shorts\n"
)
archive=Path("/content/Viral_Video_Mobile_Results.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for p in base.rglob("*"):
        if p.is_file(): z.write(p, p.relative_to(base.parent))
print(f"\nFound {len(results)} licensed downloadable files.")
print("These are free-source videos, not validated viral trends.")
files.download(str(archive))
